# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain


Разработка RAG-пайплайна


**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


**Проект выполнили:** Алмазова Диана, Воробьева Екатерина, Сысоева Анна

## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import re

# путь к файлу датасета
file_path = "arxiv-metadata-oai-snapshot.json"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "Cornell-University/arxiv",
  file_path,
  pandas_kwargs={"lines": True, "nrows": 1000}, # указываем число строк
)
df["id"] = df["id"].apply(lambda x: str(x).zfill(9))

/tmp/ipykernel_4631/534762547.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 4.82G/4.82G [04:40<00:00, 18.4MB/s]


In [ ]:
# заглянем
df.head(2)

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"


### Подбор статей

внимание на столбец id: по нему можно добраться до самой статьи

 https://arxiv.org/abs/{id}: посмотреть страницу статьи с абстрактом

 https://arxiv.org/pdf/{id}: скачать

In [ ]:
# отфильтруем df по заданному признаку и получим список id для дальнейшего сохранения
ids = df[df['categories'].str.contains('cs', na=False, regex=True)]['id'].tolist() #отбираем статьи по теме Computer Science, собираем в список
print(len(ids)) #выводим количество отобранных id

133


### Загрузка

In [ ]:
!pip install langchain_community langchain_text_splitters pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# загружаем с PyPDFLoader, используя https://arxiv.org/pdf/{id}
def load_articles(ids, max_papers = 20): #определяем функцию для загрузки статей (выбрано оптимальное количество статей для последующего разбиения на чанки, раннее указанные большие количества сильно нагружали систему + было много статей, которые не загружались)
  documents = []
  loads = 0

  for id in (ids[:max_papers]):
      try:
        url = f"https://arxiv.org/pdf/{id}"
        loader = PyPDFLoader(url)
        docs = loader.load()
        documents.extend(docs)
        loads += 1

      except Exception as e:
         print(f"Ошибка загрузки {id}: {e}")
         continue
  print(loads) #количество загрузок
  return documents
# получаем список Document объектов
documents = load_articles(ids, max_papers=20)


ERROR:pypdf._cmap:Advanced encoding [] not implemented yet


Ошибка загрузки 00704.005: Check the url of your file; returned status code 404
19


In [ ]:
print(documents[0]) #заглянуть в документ

page_content='Sparsity-certifying Graph Decompositions
Ileana Streinu1∗, Louis Theran2
1 Department of Computer Science, Smith College, Northampton, MA. e-mail:streinu@cs.smith.edu
2 Department of Computer Science, University of Massachusetts Amherst. e-mail:theran@cs.umass.edu
Abstract. We describe a new algorithm, the(k, ℓ)-pebble game with colors, and use it to obtain a charac-
terization of the family of(k, ℓ)-sparse graphs and algorithmic solutions to a family of problems concern-
ing tree decompositions of graphs. Special instances of sparse graphs appear in rigidity theory and have
received increased attention in recent years. In particular, our colored pebbles generalize and strengthen
the previous results of Lee and Streinu [12] and give a new proof of the Tutte-Nash-Williams characteri-
zation of arboricity. We also present a new decomposition that certiﬁes sparsity based on the(k, ℓ)-pebble
game with colors. Our work also exposes connections between pebble game algorithms an

In [ ]:
#конкатенация
def combine_document_fields(doc, df_metadata):
    article_id = doc.metadata.get('id')
    article_data = df_metadata[df_metadata['id'] == article_id]
    if not article_data.empty:
      combined_text = f"""
Title: {article_data['title'].iloc[0]}
Authors: {article_data['authors'].iloc[0]}
Categories: {article_data['categories'].iloc[0]}
Submitter: {article_data['submitter'].iloc[0] if 'submitter' in article_data.columns else 'N/A'}
Abstract: {article_data['abstract'].iloc[0] if 'abstract' in article_data.columns else 'N/A'}

Content:
{doc.page_content}
"""
      return combined_text
    else:
      return doc.page_content

for i, doc in enumerate(documents):
    doc.page_content = combine_document_fields(doc, df)
    doc.metadata['combined_fields'] = True


In [ ]:
#Очиcтка текста при помощи регулярных выражений
import re
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\s*\d+\s*\n', ' ', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = ' '.join(text.split())

    return text.strip()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)
print(documents[0])

page_content='Sparsity-certifying Graph Decompositions Ileana Streinu1, Louis Theran2 1 Department of Computer Science, Smith College, Northampton, MA. e-mail:streinucs.smith.edu 2 Department of Computer Science, University of Massachusetts Amherst. e-mail:therancs.umass.edu Abstract. We describe a new algorithm, the(k, )-pebble game with colors, and use it to obtain a charac- terization of the family of(k, )-sparse graphs and algorithmic solutions to a family of problems concern- ing tree decompositions of graphs. Special instances of sparse graphs appear in rigidity theory and have received increased attention in recent years. In particular, our colored pebbles generalize and strengthen the previous results of Lee and Streinu 12 and give a new proof of the Tutte-Nash-Williams characteri- zation of arboricity. We also present a new decomposition that certies sparsity based on the(k, )-pebble game with colors. Our work also exposes connections between pebble game algorithms and previou

## Разбить на чанки

In [ ]:
# выбрать тип text splitter'а под нашу задачу
# например, по статье: https://www.geeksforgeeks.org/artificial-intelligence/text-splitter-in-langchain/
# использовать split_documents

# использовать chunk_size, chunk_overlap для настройки
text_splitter = RecursiveCharacterTextSplitter(  #выбираем этот тип, так как есть возможность сплитить разные типы текста, что релевантно для статей, содержащих таблицы, формулы и пояснения к рисункам и т. д.
    chunk_size=500,
    chunk_overlap=100 #берем перекрытие 20% для сохранения контекста
)
# сохранить в chunks
chunks = text_splitter.split_documents(documents)

In [ ]:
print(len(chunks)) #выводим количество чанков
for i in range (3): #смотрим на примеры получившихся чанков
  print(f"\n Чанк {i+1}")
  print(chunks[i].metadata)
  print(chunks[i].page_content[:700])

2725

 Чанк 1
{'producer': 'pdfTeX-1.40.4', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-11-02T21:44:38+00:00', 'author': '', 'keywords': '', 'moddate': '2018-11-02T21:44:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592-1.40.5-2.2 (Web2C 7.5.6) kpathsea version 3.5.6', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/0704.0002', 'total_pages': 18, 'page': 0, 'page_label': '1', 'combined_fields': True}
Sparsity-certifying Graph Decompositions Ileana Streinu1, Louis Theran2 1 Department of Computer Science, Smith College, Northampton, MA. e-mail:streinucs.smith.edu 2 Department of Computer Science, University of Massachusetts Amherst. e-mail:therancs.umass.edu Abstract. We describe a new algorithm, the(k, )-pebble game with colors, and use it to obtain a charac- terization of the family of(k, )-sparse graphs and algorithmic solutions to a family of problems concern- ing tree decompositions of

 Чанк 2
{'producer': 'pdfTeX

## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [ ]:
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# выбираем модель
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5}) #берем количество чанков выше минимального, так как тематика статей достаточно сложная, необходимо дать больше контекста
print(f"Обработано чанков: {vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Обработано чанков: 2725


## Реализовать цепочку

In [ ]:
!pip -q install -U datasets langchain langchain-huggingface langchain-groq faiss-cpu sentence-transformers


In [ ]:
import json
import os
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

In [ ]:
os.environ["GROQ_API_KEY"] = "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" #работает с VPN
llm = ChatGroq(
    model="llama-3.3-70b-versatile", #берем эту модель, так как более ранняя версия устарела и не поддерживается, а новая не распознает какие-то из вопросов :(
    temperature=0,
    max_retries=2
)
prompt = ChatPromptTemplate.from_template("""You are a helpful research assistant specializing in computer science papers.
Answer the question based ONLY on the following context from academic papers.

RULES:
1. Use ONLY information from the context provided below
2. If the context doesn't contain the answer, say: "I don't have enough information to answer this question based on the provided papers."
3. If you use information from multiple sources, cite them
4. Be concise but comprehensive

Context from research papers:
{context}

Question: {question}

Provide a clear, accurate answer based ONLY on the context above:""")

def join_docs(docs): #объединяем документы в одну строку
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = ( #создаем цепочку
    {
        "context": retriever | join_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)


In [ ]:
questions = [
  ("factual question", "What is the main point of the paper titled Sparsity-certifying Graph Decompositions"),
  ("general question", "What computer science methods were mentioned in retrieved papers?"), #Вопрос оказался сложным для модели, иногда она не выдавала ответ на него
  ("specific question", "How the concept of a graph is used in the article in the paper titled Sparsity-certifying Graph Decompositions?"),
  ("out_of_context", "What is the capital of Сhina?")  # Вопрос проверяет, как модель работает с отсутствием информации в контексте
]

for question_type, question in questions:
        print("\n" + "="*80)
        print(f"Тип вопроса: {question_type}")
        print(f"Вопрос: {question}")

        response = rag_chain.invoke(question)
        used_chunks = retriever.invoke(question)

        print(f"\nОтвет: {response.content}")
        for i, doc in enumerate(used_chunks, start=1):
            print(f"\n Чанк {i}")
            print(f"Метаданные: {doc.metadata}")
            print(f"\nТекст чанка: {doc.page_content[:500]}")



Тип вопроса: factual question
Вопрос: What is the main point of the paper titled Sparsity-certifying Graph Decompositions

Ответ: The main point of the paper titled Sparsity-certifying Graph Decompositions is to introduce a new characterization of sparse graphs, called the pebble game with colors, which produces a sparse graph along with a sparsity-certifying decomposition, and to present algorithms based on this characterization that apply to the study of sparse graphs and their decompositions (as mentioned in the context, but no specific citation is provided in the given context).

 Чанк 1
Метаданные: {'producer': 'pdfTeX-1.40.4', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-11-02T21:44:38+00:00', 'author': '', 'keywords': '', 'moddate': '2018-11-02T21:44:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592-1.40.5-2.2 (Web2C 7.5.6) kpathsea version 3.5.6', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/0704.0002', 'tot

## Протестировать на нескольких примерах, оченить качество

1. На первый вопрос модель дала в целом правильный ответ с упоминанием предмета исследования, цели и ключевых понятий.  Основным для ответа стал пятый чанк, содержащий информацию о значении характеристики "the pebble game with colors", остальные чанки в большей степени посвящены описаниям самих графов. Были использованы фрагменты текста только из указанной в вопросе статьи.
2. Со вторым вопросом возникало больше всего проблем, так как модель в половине случаев его не обрабатывала, выводя в ответе, что контекст не содержит достаточно информации. Возможно, это связано с тем, что методы не отмечались эксплицитно как методы computer science, а в контекстах, где присутствовали слова computer или science, не содержали релевантные тезисы. Тем не менее модель выдала три метода (как кажется, они находятся в разной степени обобщенности) и очертила области, в которых методология применима. Для подготовки ответа были использованы чанки из четырех разных статей, и в них есть упоминания об этих методах.
3. На третий вопрос модель дала исчерпывающий и подробный ответ о том, как используется понятие графа в интересующей статье, и привела несколько важных характеристик графов. Однако, для ответа на вопрос были использованы те же самые чанки, что и для первого вопроса, но в этом случае ответ кажется более полным, а чанки - более подходящими для заданного вопроса.
4. Был задан вопрос, об ответе на который не было информации в контексте, и модель ожидаемо на него не ответила. В чанках находятся различные имена и фамилии.

Таким образом, модель дает ответы на поставленные вопросы. При этом необходимо учесть, что: 1) результат может быть хаотичным (например, методы во втором вопросе описаны не системно) 2) модель может использовать чанки, в которых присутствуют слова из вопроса, но они не содержат информации по существу.
Можно выделить следующие возможности для улучшения:
1) Добавление к вопросу ключевых слов. Тематика выбранных работ техническая и достаточно сложная, опора на ключевые слова может помочь ориентироваться в этом поле;
2) Добавление тематических тезисов в промт, которые вводили бы в тематику изучаемой области.




---



# Критерии оценки

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран критерий подбора материалов
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные приведены к формату, пригодному для чанкинга (очистка, объединение полей)

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная эмбеддинг-модель (например, all-MiniLM-L6-v2 для русского/английского)
- [ ] 0.5 балла: индекс создан


### Реализация цепочки (3 балла)
- [ ] 0.5 балла: выбрана LLM
- [ ] 1 балл: промпт, QUESTION, TASK составлены корректно
- [ ] 0.5 балла: обоснован заданный TOP_K
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)
- [ ] 0.5 балла: обработан случай отсутствия информации в контексте

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
